# GraphSentinel v2 — Colab Execution Runbook

**Run top to bottom.** Every cell says why it exists in its first comment.

---

## Read this before you run anything

**What this model is.** Nodes are **IP addresses / hosts**. Edges are **network
flows** between them. `prediction[i]` is a **host**, identified by
`data.node_ip_int[i]` — so it maps directly to an IP you can block. There is
also a second head that classifies **every individual flow**, which is what
produces an OpenFlow drop rule with an exact `(src, dst, proto, port)` match.

This matters because it is the one thing v1 got wrong: v1 made each *flow row* a
node and invented edges between consecutive CSV rows. That graph could not
answer "which host do I block" without a manual aggregation layer, and it had no
network topology in it at all.

**What is proven, and what is not:**

| claim | status |
|---|---|
| Nodes are hosts, edges are flows, predictions map to IPs | **proven by code + tests** |
| Graph encodes attack topology (scan fan-out, DDoS fan-in) | **proven by tests** — asserted against the 99th percentile of other hosts |
| Model trains and detects all 5 attack families | **empirically validated on SYNTHETIC traffic only** |
| Same on real CICIDS2017 | **not yet proven — this notebook is how you prove it** |
| Real-time latency on Mininet/Scapy traffic | **not empirically verified** — Section 10 measures it |
| Zero-day / unknown attack detection | **weak.** See Section 11. Do not market this as a zero-day detector. |
| Safe for autonomous blocking | **NO.** SDN translator defaults to `dry_run=True`. Keep it there. |

**On "self-healing":** the model emits detections and *proposed* OpenFlow rules.
It does not block anything. That boundary is deliberate — an IDS with write
access to the network is a denial-of-service tool when it is wrong, and it will
sometimes be wrong.

---

## Colab crash safety (read this once)

Colab kills the runtime when RAM fills, and **everything in `/content` is lost**.
This notebook is built so that never costs you more than a couple of minutes:

| what | where it lives | survives restart? | cost to rebuild |
|---|---|---|---|
| Model checkpoints (`last.pt`, `best.pt`) | **Google Drive** | **yes** | — |
| Parquet splits | **Google Drive** | **yes** | ~1–3 min |
| Built graph cache (the big one) | local disk by default | no | ~1–2 min |
| Logs, model card, exports | **Google Drive** | **yes** | — |

`train(..., resume=True)` picks up from the last completed **epoch**, not from
scratch. After a crash: re-run cells 1–6, then jump straight back to the
training cell.

**If results look wrong**, use the RESET cell (Section 4). It is granular and
requires you to type `RESET` to confirm, so you cannot nuke a long run by
accidentally re-running a cell.

## 1 — Environment

In [ ]:
# WHY: mount Drive and locate the PROJECT ROOT.
# The critical distinction, and the #1 cause of "ModuleNotFoundError:
# graphsentinel" on Colab:
#
#   PROJECT ROOT (this is BASE)  ->  contains graphsentinel/, tests/, datasets/
#   PYTHON PACKAGE               ->  the INNER graphsentinel/ holding config.py
#
# The release zip is nested (graphsentinel/graphsentinel/config.py), so if you
# unzipped and uploaded the whole thing, your real BASE is one level deeper than
# you expect. This cell finds the folder where BASE/graphsentinel/config.py
# actually exists, rather than trusting a folder name.
from google.colab import drive
import os, sys, glob

drive.mount('/content/drive')

MYDRIVE = "/content/drive/MyDrive"

def is_project_root(d):
    """A real project root holds the PACKAGE: d/graphsentinel/config.py"""
    return os.path.isfile(os.path.join(d, "graphsentinel", "config.py"))

# search order: usual names, then one level under each, then a shallow scan
candidates = [
    os.path.join(MYDRIVE, n)
    for n in ["GraphSentinel", "GraphSentinelV2", "graphsentinel", "GraphSentinel_v2"]
]
candidates += [os.path.join(c, "graphsentinel") for c in list(candidates)]
candidates += sorted(glob.glob(os.path.join(MYDRIVE, "*")))
candidates += sorted(glob.glob(os.path.join(MYDRIVE, "*", "*")))

BASE = next((c for c in candidates if os.path.isdir(c) and is_project_root(c)), None)

if BASE is None:
    print("COULD NOT FIND THE PROJECT ROOT.")
    print()
    print("Looking for a folder X where X/graphsentinel/config.py exists.")
    print()
    print("What is actually in your Drive:")
    for d in sorted(glob.glob(os.path.join(MYDRIVE, "*")))[:25]:
        if os.path.isdir(d):
            try:
                inner = sorted(os.listdir(d))[:6]
            except OSError:
                inner = ["<unreadable>"]
            print("  {:<32s} -> {}".format(os.path.basename(d), inner))
    raise SystemExit(
        "Fix the upload so <BASE>/graphsentinel/config.py exists, then re-run."
    )

print("PROJECT ROOT (BASE) =", BASE)
print("python package      =", os.path.join(BASE, "graphsentinel"))

sys.path.insert(0, BASE)
print()
print("contents of BASE:", sorted(os.listdir(BASE)))
print()

for need, why in [
    ("graphsentinel", "the Python package (must contain config.py)"),
    ("tests", "make_synthetic.py, used by the synthetic-data cell"),
]:
    ok = os.path.isdir(os.path.join(BASE, need))
    print("  {} {:<16s} {}".format("OK     " if ok else "MISSING", need + "/", why))

DATA_DIR = os.path.join(BASE, "datasets", "cicids2017")
os.makedirs(DATA_DIR, exist_ok=True)
n_csv = len(glob.glob(os.path.join(DATA_DIR, "*.csv")))
n_zip = len(glob.glob(os.path.join(DATA_DIR, "*.zip")))
print("  OK      {:<16s} {} CSV, {} zip".format("datasets/cicids2017/", n_csv, n_zip))
if n_csv == 0 and n_zip == 0:
    print()
    print("  No data yet -> use the unzip cell (2A) or the synthetic cell (2B).")

In [ ]:
# WHY: confirm the T4 is actually attached before spending time on setup.
# If this prints CPU, fix it now: Runtime > Change runtime type > T4 GPU.
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used",
                      "--format=csv"], capture_output=True, text=True).stdout or "no nvidia-smi")
!free -g | head -2
!df -h /content | tail -1

In [ ]:
# WHY: install the packages Colab does not ship. torch itself is NOT installed
# here on purpose -- Colab's preinstalled torch is CUDA-matched to the runtime,
# and pip-installing over it is the single most common way to break the GPU.
!pip install -q torch-geometric pyarrow
!pip install -q onnx onnxscript onnxruntime      # optional: ONNX export
!pip install -q fastapi uvicorn pydantic          # optional: inference service
print("done")

In [ ]:
# WHY: copy the package from Drive to local disk before importing it.
# Drive is a network filesystem -- importing ~20 modules across it on every cell
# is slow and occasionally flaky. Data and checkpoints still live on Drive; only
# the CODE is local. Re-run this cell after editing any file on Drive.
import shutil, sys, os, importlib

CODE_DIR = "/content/gs_code"
shutil.rmtree(CODE_DIR, ignore_errors=True)
os.makedirs(CODE_DIR, exist_ok=True)
shutil.copytree(os.path.join(BASE, "graphsentinel"), os.path.join(CODE_DIR, "graphsentinel"))
if os.path.isdir(os.path.join(BASE, "tests")):
    shutil.copytree(os.path.join(BASE, "tests"), os.path.join(CODE_DIR, "tests"))

for p in (CODE_DIR, os.path.join(CODE_DIR, "tests")):
    if p in sys.path:
        sys.path.remove(p)
    sys.path.insert(0, p)

# drop any already-imported copy so a re-run picks up edited files
for mod in [m for m in list(sys.modules) if m.startswith("graphsentinel")]:
    del sys.modules[mod]

print("code copied to", CODE_DIR)

In [ ]:
# WHY: fail fast on a broken install. If PyG or the package is wrong, we want
# to know here -- not 40 minutes into a training run.
import torch, torch_geometric, graphsentinel
from graphsentinel.config import Config, CLASS_NAMES
from graphsentinel.data.graph_builder import EDGE_FEATURE_NAMES, NODE_FEATURE_NAMES

print("torch          :", torch.__version__)
print("torch-geometric:", torch_geometric.__version__)
print("graphsentinel  :", graphsentinel.__version__)
print("CUDA available :", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print()
print("classes        :", CLASS_NAMES)
print("node features  :", len(NODE_FEATURE_NAMES), "(a HOST's structural profile)")
print("edge features  :", len(EDGE_FEATURE_NAMES), "(one FLOW's attributes)")

## 2 — Data

Two paths. **Pick one.**

- **2A Real CICIDS2017** — the one that matters. You need
  `GeneratedLabelledFlows.zip`, *not* `MachineLearningCSV.zip`. The two ship with
  identical filenames but only the first has Source IP / Destination IP /
  Timestamp columns, and without those an IP-as-node graph cannot be built at all.
- **2B Synthetic** — CICIDS-shaped traffic with real attack topology, generated
  in seconds. Use it to verify the pipeline end-to-end before committing to a
  long real-data run. Numbers from it are **not** publishable results.

In [ ]:
# WHY (2A step 1): unzip GeneratedLabelledFlows.zip and FLATTEN it.
# The archive extracts into a "TrafficLabelling_/" subfolder, but the loader
# expects the CSVs directly inside datasets/cicids2017/. Skipping the flatten
# is why "I unzipped it but it says no files found" happens.
import glob, os, shutil

DATA_DIR = os.path.join(BASE, "datasets", "cicids2017")
zips = glob.glob(os.path.join(DATA_DIR, "*.zip"))
print("zip files found:", [os.path.basename(z) for z in zips])

for z in zips:
    print("extracting", os.path.basename(z), "...")
    shutil.unpack_archive(z, DATA_DIR)

# flatten: move every CSV found at any depth up into DATA_DIR
moved = 0
for src in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    dst = os.path.join(DATA_DIR, os.path.basename(src))
    if os.path.abspath(src) != os.path.abspath(dst):
        shutil.move(src, dst)
        moved += 1
print(f"flattened {moved} CSV(s)")

for f in sorted(glob.glob(os.path.join(DATA_DIR, "*.csv"))):
    print(f"  {os.path.basename(f):<58s} {os.path.getsize(f)/1e6:7.1f} MB")
print("\nTip: delete the .zip from Drive now to save quota.")

In [ ]:
# WHY (2A step 2): verify you have the RIGHT CICIDS2017 distribution before
# training. This reads only the header row of each CSV. If you have the
# MachineLearningCVE variant it raises with download instructions instead of
# silently degrading into a meaningless graph.
from graphsentinel.data.schema import validate_dataset, SchemaError
from graphsentinel.config import Config

_probe = Config(); _probe.base_dir = BASE
try:
    reports = validate_dataset(_probe.dataset_path, _probe.data.csv_files,
                               require_ips=True, verbose=True)
    print("\nSCHEMA OK -- IP + timestamp columns present. Real-data training is possible.")
    USING_REAL_DATA = True
except SchemaError as e:
    USING_REAL_DATA = False
    print(e)
    print("\n>>> Either fix the dataset, or use the synthetic cell below to proceed.")

In [ ]:
# WHY (2B): generate synthetic CICIDS-shaped traffic. Run this ONLY if you do
# not have the real data yet, or to smoke-test the whole pipeline fast.
# It writes the same five filenames, so everything downstream is identical.
#
# SET THIS TO True TO OVERWRITE YOUR DATASET FOLDER WITH SYNTHETIC DATA.
GENERATE_SYNTHETIC = False

if GENERATE_SYNTHETIC:
    from make_synthetic import write_dataset
    out = write_dataset(os.path.join(BASE, "datasets", "cicids2017"), seed=7)
    print("wrote synthetic dataset to", out)
    for f in sorted(glob.glob(os.path.join(str(out), "*.csv"))):
        print(f"  {os.path.basename(f):<58s} {os.path.getsize(f)/1e6:6.2f} MB")
    USING_REAL_DATA = False
else:
    print("skipped (GENERATE_SYNTHETIC = False)")

## 3 — Configuration

In [ ]:
# WHY: one place for every knob, and the crash-safety decision.
#
# LITE_MODE trades coverage for RAM/time. Turn it ON for your first real-data
# run so you find problems in 10 minutes instead of an hour.
#
# GRAPH_CACHE_ON_DRIVE is the crash-safety tradeoff:
#   False (default) -> graph cache on fast local disk. Lost on restart, but
#                      rebuilding takes ~1-2 min because the expensive step
#                      (CSV parsing) is cached separately as parquet on Drive.
#   True            -> everything on Drive. Survives restart, but the cache can
#                      be several GB and Drive writes are slow.
# Checkpoints, logs and exports ALWAYS go to Drive either way.

from graphsentinel.config import Config

LITE_MODE            = True
GRAPH_CACHE_ON_DRIVE = False
EPOCHS               = 40
WINDOW_SECONDS       = 60
SPLIT_STRATEGY       = "episode"   # episode | temporal | host_holdout | attack_holdout

cfg = Config()
cfg.base_dir                 = BASE
cfg.graph.window_seconds     = WINDOW_SECONDS
cfg.graph.window_stride_seconds = WINDOW_SECONDS
cfg.data.split_strategy      = SPLIT_STRATEGY
cfg.train.epochs             = EPOCHS
cfg.train.seed               = 42
cfg.model.recon_enabled      = True     # joint self-supervised anchor

if LITE_MODE:
    # Two capture days instead of five, and wider windows -> fewer, bigger graphs.
    cfg.data.csv_files = [
        "Tuesday-WorkingHours.pcap_ISCX.csv",
        "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    ]
    cfg.graph.window_seconds = cfg.graph.window_stride_seconds = 120
    cfg.graph.max_edges_per_graph = 50_000
    print("LITE_MODE ON -- 2 capture days, 120s windows.")
    print("  NOTE: this is a PIPELINE SMOKE TEST, not a result. Two days means")
    print("  only 2 attack families exist, and the model will be degenerate.")
    print("  Set LITE_MODE = False for any number you intend to report.")

if not GRAPH_CACHE_ON_DRIVE:
    # absolute path overrides base_dir, so the bulky cache stays off Drive
    cfg.data.processed_dir = "/content/gs_processed/"

cfg.ensure_dirs()
cfg.save(cfg.log_path / "config.json")

print("\nbase       :", cfg.base_dir)
print("dataset    :", cfg.dataset_path)
print("processed  :", cfg.processed_path, "  <- graph cache")
print("checkpoints:", cfg.checkpoint_path, "  <- SURVIVES RESTART")
print("models     :", cfg.model_path)
print("logs       :", cfg.log_path)
print("\nsplit:", cfg.data.split_strategy, "| window:", cfg.graph.window_seconds,
      "s | epochs:", cfg.train.epochs)

## 4 — Checkpoint state & RESET

Run **4A** any time to see where you are. Run **4B** only when you want to throw
work away.

In [ ]:
# WHY (4A): show exactly what state exists on disk and where a resume would
# pick up from. Run this after a crash to see what survived.
import os, glob, torch
from pathlib import Path

def _size(p):
    p = Path(p)
    if p.is_file():
        return p.stat().st_size
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())

def _human(n):
    for u in ["B", "KB", "MB", "GB"]:
        if n < 1024: return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} TB"

print("STATE")
print("-" * 66)
for label, path in [("checkpoints", cfg.checkpoint_path),
                    ("processed  ", cfg.processed_path),
                    ("models     ", cfg.model_path),
                    ("logs       ", cfg.log_path)]:
    exists = Path(path).exists()
    n = len(list(Path(path).glob("*"))) if exists else 0
    print(f"  {label}  {'exists' if exists else 'missing':>7s}  {n:>3d} file(s)  "
          f"{_human(_size(path)) if exists else '-':>10s}")

last = Path(cfg.checkpoint_path) / "last.pt"
if last.exists():
    ck = torch.load(last, map_location="cpu", weights_only=False)
    print(f"\n  RESUME POINT: epoch {ck['epoch']}  "
          f"best {cfg.train.early_stop_metric} = {ck.get('best_metric', float('nan')):.4f}")
    print(f"  -> train(resume=True) will continue from epoch {ck['epoch'] + 1}")
else:
    print("\n  no checkpoint -- training will start from epoch 1")

for f in sorted(Path(cfg.processed_path).glob("*")) if Path(cfg.processed_path).exists() else []:
    print(f"  cache: {f.name:<44s} {_human(f.stat().st_size)}")

In [ ]:
# WHY (4B): throw away corrupted or unwanted state. GRANULAR on purpose --
# rebuilding the graph cache is cheap, but re-running 40 epochs is not, so you
# rarely want to delete everything.
#
# Nothing happens unless CONFIRM == "RESET". Set the flags, type the word, run.
#
# Typical uses:
#   results look wrong / loss went NaN  -> RESET_CHECKPOINTS + RESET_LOGS
#   changed window_seconds or split     -> RESET_GRAPH_CACHE + RESET_PROCESSED
#   changed the dataset files           -> RESET_PROCESSED (forces CSV re-parse)
#   starting completely fresh           -> all True

RESET_CHECKPOINTS = False   # last.pt / best.pt      -> loses training progress
RESET_GRAPH_CACHE = False   # graphs_*.pt            -> ~1-2 min to rebuild
RESET_PROCESSED   = False   # *_<split>.parquet      -> ~1-3 min to rebuild
RESET_LOGS        = False   # training_log.csv, test_report.json
RESET_MODELS      = False   # weights.pt, model_card.json, onnx/ts exports

CONFIRM = ""                # <-- type RESET here to arm

import shutil
from pathlib import Path

if CONFIRM != "RESET":
    print('DISARMED. Nothing deleted. Set CONFIRM = "RESET" to actually delete.')
else:
    deleted = []
    if RESET_CHECKPOINTS:
        for f in Path(cfg.checkpoint_path).glob("*.pt"):
            f.unlink(); deleted.append(str(f))
    if RESET_GRAPH_CACHE:
        for f in Path(cfg.processed_path).glob("graphs_*.pt"):
            f.unlink(); deleted.append(str(f))
    if RESET_PROCESSED:
        for f in Path(cfg.processed_path).glob("*.parquet"):
            f.unlink(); deleted.append(str(f))
    if RESET_LOGS:
        for f in Path(cfg.log_path).glob("*"):
            if f.is_file(): f.unlink(); deleted.append(str(f))
    if RESET_MODELS:
        for f in Path(cfg.model_path).glob("*"):
            if f.is_file(): f.unlink(); deleted.append(str(f))

    cfg.ensure_dirs()
    print(f"deleted {len(deleted)} file(s):")
    for d in deleted:
        print("  -", d)
    if not deleted:
        print("  (nothing matched -- flags may all be False)")

## 5 — Build splits and graphs

In [ ]:
# WHY: parse the CSVs, split them without leaking an attack burst across
# train/test, and turn each time window into a host graph. Cached -- re-running
# is instant. This is the RAM-heaviest step, so it reports memory as it goes.
import time, json, psutil, gc

def ram():
    v = psutil.virtual_memory()
    return f"RAM {v.used/1e9:.1f}/{v.total/1e9:.1f} GB ({v.percent:.0f}%)"

print("before:", ram())
t0 = time.time()

from graphsentinel.train import prepare_graphs
from graphsentinel.data.graph_builder import summarise_graphs

graphs = prepare_graphs(cfg, force=False, verbose=True)

gc.collect()
print(f"\nbuilt in {time.time()-t0:.1f}s |", ram())
for name, gs in graphs.items():
    s = summarise_graphs(gs, cfg.model.num_classes)
    print(f"\n{name}: {len(gs)} graphs")
    if s:
        print(f"  nodes/graph avg {s['nodes_mean']:.0f} max {s['nodes_max']}")
        print(f"  edges/graph avg {s['edges_mean']:.0f} max {s['edges_max']}")
        print(f"  node classes {dict(zip(CLASS_NAMES, s['node_class_counts']))}")

In [ ]:
# WHY: prove the graph actually encodes attack topology BEFORE training.
# If a port scanner does not stand out on destination-port entropy, the graph is
# wrong and no amount of model tuning will rescue it. Cheap check, saves hours.
import numpy as np, matplotlib.pyplot as plt
from graphsentinel.data.graph_builder import NODE_FEATURE_NAMES

WATCH = ["dst_port_entropy", "log_unique_src_ips", "log_unique_dst_ips", "mean_inter_flow_dt"]
vals = {n: [] for n in WATCH}
labels = []
for g in graphs["train"]:
    labels.append(g.y.numpy())
    for n in WATCH:
        vals[n].append(g.x[:, NODE_FEATURE_NAMES.index(n)].numpy())
labels = np.concatenate(labels)

fig, axes = plt.subplots(1, len(WATCH), figsize=(5 * len(WATCH), 4))
for ax, (name, v) in zip(np.atleast_1d(axes), vals.items()):
    v = np.concatenate(v)
    data = [v[labels == i] if (labels == i).any() else np.array([0.0])
            for i in range(len(CLASS_NAMES))]
    ax.boxplot(data, tick_labels=[c[:6] for c in CLASS_NAMES], showfliers=False)
    ax.set_title(name, fontsize=10); ax.tick_params(axis="x", rotation=45); ax.grid(alpha=.3)
plt.suptitle("Structural host features by class -- separation here is the whole point",
             fontweight="bold")
plt.tight_layout(); plt.show()

print("Expect: PortScan high on dst_port_entropy; DDoS high on log_unique_src_ips.")
print("If the boxes overlap completely, STOP and check the data before training.")

## 6 — Train

**This is the long cell.** It writes a full checkpoint to Drive after *every
epoch*, so a Colab OOM kill costs you at most one epoch.

**If the runtime dies:** re-run cells 1→6 (mount, GPU, install, copy code,
imports, config), then re-run this cell. `resume=True` continues from the last
completed epoch.

In [ ]:
# WHY: train the model. resume=True means a crashed run picks up where it
# stopped instead of starting over. force_rebuild=False reuses the graph cache.
#
# Watch: macroF1 is the headline, NOT accuracy. Accuracy on this class balance is
# beaten by "guess benign". Watch the per-class numbers -- Botn (botnet) and SSHB
# (brute force) are the hard minority classes and the ones that reveal a model
# that is quietly failing.
import time
from graphsentinel.train import train

t0 = time.time()
report = train(cfg, resume=True, force_rebuild=False, verbose=True)
print(f"\ntotal wall clock: {(time.time()-t0)/60:.1f} min")

In [ ]:
# WHY: plot what happened. If val macro F1 is flat near zero while train loss
# falls, the model is memorising; if the per-class lines for Botnet/SSHBrute
# never leave the floor, the minority classes are not being learned.
import pandas as pd, matplotlib.pyplot as plt

hist = pd.read_csv(cfg.log_path / "training_log.csv")
fig, axes = plt.subplots(1, 3, figsize=(19, 5))

axes[0].plot(hist.epoch, hist.train_loss, lw=2, label="train loss")
if "recon_loss" in hist and hist.recon_loss.abs().sum() > 0:
    axes[0].plot(hist.epoch, hist.recon_loss, lw=1.5, ls="--", label="recon loss")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=.3)

for k, lbl in [("node_macro_f1", "node macro F1"), ("edge_macro_f1", "edge macro F1"),
               ("node_binary_pr_auc", "binary PR-AUC")]:
    if k in hist: axes[1].plot(hist.epoch, hist[k], lw=2, label=lbl)
axes[1].set_title("Validation"); axes[1].set_ylim(0, 1.02)
axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].grid(alpha=.3)

for c in CLASS_NAMES:
    k = f"node_f1_{c}"
    if k in hist: axes[2].plot(hist.epoch, hist[k], lw=1.8, label=c)
axes[2].set_title("Per-class F1 -- watch the minority classes")
axes[2].set_ylim(0, 1.02); axes[2].set_xlabel("epoch")
axes[2].legend(fontsize=8); axes[2].grid(alpha=.3)

plt.tight_layout(); plt.savefig(cfg.log_path / "training_curves.png", dpi=140)
plt.show()
print("saved ->", cfg.log_path / "training_curves.png")

## 7 — Test results

In [ ]:
# WHY: read the held-out test numbers. Reported metrics are deliberately NOT
# accuracy/weighted-F1 -- both are satisfiable by a model that never detects a
# botnet. recall_at_fpr_0.01 is the operational number: how much you catch when
# false alerts are capped at 1% of benign hosts.
import json, numpy as np, matplotlib.pyplot as plt

rep = json.load(open(cfg.log_path / "test_report.json"))
m = rep["metrics"]

print("=" * 64)
print(f"  TEST -- split protocol: {cfg.data.split_strategy}")
print("=" * 64)
for k in ["node_macro_f1", "edge_macro_f1", "node_binary_pr_auc",
          "node_recall_at_fpr_0.01", "node_recall_at_fpr_0.001", "node_accuracy"]:
    if k in m: print(f"  {k:<28s} {m[k]:.4f}")

print("\n  per-class node F1 (a 0.0000 means that attack is NOT detected):")
for c in CLASS_NAMES:
    v = m.get(f"node_f1_{c}", float("nan"))
    flag = "  <-- NOT DETECTED" if v < 0.05 else ""
    print(f"    {c:<10s} {v:.4f}{flag}")

try:
    import seaborn as sns
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    for ax, kind in zip(axes, ["node", "edge"]):
        cm = np.array(rep[f"{kind}_confusion"])
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
        ax.set_title(f"{kind}-level confusion"); ax.set_xlabel("predicted"); ax.set_ylabel("true")
    plt.tight_layout(); plt.show()
except Exception as e:
    print("plot skipped:", e)

In [ ]:
# WHY: evasion check. Flow Bytes/s and Flow Packets/s are under an attacker's
# direct control -- rate-limit the payload and those columns say whatever they
# want. This zeroes them and re-measures. Small deltas = the model is reading
# topology (hard to fake). Large deltas = it is reading spoofable headers.
abl = rep.get("evasion_ablation")
if abl:
    print(f"{'metric':<32s} {'baseline':>9s} {'ablated':>9s} {'delta':>9s}")
    for k, v in abl["delta"].items():
        b, a = abl["baseline"].get(k, float("nan")), abl["ablated"].get(k, float("nan"))
        print(f"  {k:<30s} {b:9.4f} {a:9.4f} {v:+9.4f}")
    worst = max(abs(v) for v in abl["delta"].values())
    print(f"\nlargest degradation: {worst:.4f}")
    print("< 0.05 -> reading structure (good). > 0.15 -> leaning on volume (fragile).")
else:
    print("no ablation in report")

## 8 — Export artefacts for the backend

In [ ]:
# WHY: write the files the backend consumes. model_card.json is the contract --
# feature names, order, classes, version -- so the backend never hard-codes a
# feature count the way v1 did with in_channels=7.
import torch, numpy as np
from graphsentinel.export import export_all, verify_export
from graphsentinel.models.net import build_model
from graphsentinel.inference.ema_scaler import EMAScaler
from graphsentinel.data.graph_builder import EDGE_FEATURE_NAMES, NODE_FEATURE_NAMES

model = build_model(cfg)
best = torch.load(cfg.checkpoint_path / "best.pt", map_location="cpu", weights_only=False)
model.load_state_dict(best["model"])
print(f"loaded best checkpoint from epoch {best['epoch']}")

# warm-start the streaming scaler from the TRAINING feature distribution
edge_x = np.concatenate([g.edge_attr.numpy() for g in graphs["train"][:200]])
node_x = np.concatenate([g.x.numpy()        for g in graphs["train"][:200]])
edge_scaler = EMAScaler.from_training(edge_x, EDGE_FEATURE_NAMES)
node_scaler = EMAScaler.from_training(node_x, NODE_FEATURE_NAMES)

status = export_all(cfg, model, cfg.model_path, edge_scaler, node_scaler,
                    metrics=rep["metrics"])
print("\nverification (exported vs eager outputs):")
print(verify_export(cfg.model_path, model))

In [ ]:
# WHY: confirm the artefacts really landed on Drive and survive this session.
from pathlib import Path
print("files in", cfg.model_path)
print("-" * 66)
for f in sorted(Path(cfg.model_path).glob("*")):
    print(f"  {f.name:<26s} {f.stat().st_size/1e6:8.2f} MB")

expected = ["weights.pt", "model_card.json", "edge_scaler.json", "node_scaler.json"]
missing = [e for e in expected if not (Path(cfg.model_path) / e).exists()]
print("\nrequired:", "ALL PRESENT" if not missing else f"MISSING {missing}")
print("optional: model.onnx / model.ts export best-effort -- absence is not a failure")

## 9 — Detection → IP → proposed block

This is the chain GraphSentinel's self-healing layer depends on. It runs the
**same code path** production uses, on replayed traffic.

In [ ]:
# WHY: prove predictions map to real IP addresses and to installable rules.
# This is the exact question the self-healing layer needs answered: which host,
# and what do I drop? Note rules are PROPOSED -- dry_run is on by default.
import numpy as np
from graphsentinel.inference.engine import InferenceEngine
from graphsentinel.inference.capture import CSVReplaySource, run_pipeline

engine = InferenceEngine.from_artifacts(cfg.model_path, threat_threshold=0.75)
csvs = sorted(Path(cfg.dataset_path).glob("*.csv"))
src = CSVReplaySource(csvs[-1], speed=0.0, limit=40_000)   # speed=1.0 = real time

results = []
run_pipeline(src, engine, batch_size=2048, on_result=results.append)

lat = [r.latency_ms for r in results] or [0]
print(f"windows closed : {len(results)}")
print(f"flows processed: {sum(r.n_flows for r in results):,}")
print(f"window latency : median {np.median(lat):.0f} ms  p95 {np.percentile(lat,95):.0f} ms")
print(f"window budget  : {cfg.graph.window_seconds*1000} ms  "
      f"-> {'KEEPS UP' if np.percentile(lat,95) < cfg.graph.window_seconds*1000 else 'TOO SLOW'}")

print("\nDETECTED HOSTS (prediction -> IP):")
shown = 0
for r in results:
    for d in r.detections[:3]:
        print(f"  {d.ip:<16s} {d.attack_class:<10s} score={d.threat_score:.3f} "
              f"out_deg={d.out_degree:<6d} in_deg={d.in_degree}")
        shown += 1
    if shown >= 15: break
if not shown:
    print("  (none above threshold -- expected if this file is mostly benign)")

In [ ]:
# WHY: show the proposed enforcement actions. Each carries a full 5-tuple and a
# finite timeout -- nothing this system proposes is permanent.
# SAFETY: dry_run defaults True. Before you ever enable it, set allow_networks to
# cover your gateways, DNS resolvers and the controller itself.
n = 0
for r in results:
    for rule in r.rules:
        print(f"{rule.attack_class:<10s} {rule.src_ip:>15s} -> {rule.dst_ip:<15s} "
              f":{rule.dst_port:<6d} proto={rule.protocol}  {rule.action:<20s} "
              f"conf={rule.confidence:.3f}")
        print("   ", rule.to_ovs_ofctl())
        n += 1
        if n >= 10: break
    if n >= 10: break

total = sum(len(r.rules) for r in results)
print(f"\ntotal proposed rules: {total}")
print(f"translator dry_run  : {engine.translator.dry_run}  (keep True until lab-validated)")
if total == 0:
    print("\nZERO RULES IS NOT NECESSARILY A BUG -- run the next cell to find out why.")

In [ ]:
# WHY: distinguish "SDN layer is broken" from "model is not confident enough
# yet". Rules are gated on per-class minimum confidence, so an undertrained
# model produces detections but NO rules -- that is the safety rail working, not
# a failure. This shows the edge-head confidence distribution against the gates.
import numpy as np, pandas as pd
from graphsentinel.inference.sdn import MITIGATION_POLICY

df = pd.read_csv(csvs[-1], low_memory=False, encoding="latin-1")
df.columns = [c.strip() for c in df.columns]
df["t"] = pd.to_datetime(df["Timestamp"], errors="coerce", dayfirst=True).astype("int64") // 10**9
df = df.dropna(subset=["t"]).sort_values("t").head(8000)
df["y"] = 0

eng2 = InferenceEngine.from_artifacts(cfg.model_path, threat_threshold=0.5)
g = eng2.builder.build(eng2._coerce(df), update_history=False, verbose=False)[0]
out = eng2.model.predict(g)

ep = out["edge_probs"].numpy()
mask = g.real_edge_mask.numpy()
pred, conf = ep[mask].argmax(1), ep[mask].max(1)

print("EDGE-HEAD predictions (these are what become rules):")
print(f"  {'class':<10s} {'edges':>8s} {'mean conf':>10s} {'max conf':>9s} {'gate':>6s} {'passes':>7s}")
for i, c in enumerate(CLASS_NAMES):
    n = int((pred == i).sum())
    if not n:
        continue
    gate = MITIGATION_POLICY.get(c, {}).get("min_conf")
    passes = int((conf[pred == i] >= gate).sum()) if gate else 0
    print(f"  {c:<10s} {n:>8d} {conf[pred==i].mean():>10.3f} {conf[pred==i].max():>9.3f} "
          f"{str(gate) if gate else '-':>6s} {passes:>7d}")

npb = out["node_probs"].numpy()
print(f"\nNODE head: {(npb.argmax(1) > 0).sum()}/{len(npb)} hosts flagged, "
      f"max threat {(1 - npb[:, 0]).max():.3f}")
print("\nHow to read this:")
print("  edges flagged as attack but 0 passing the gate -> model undertrained.")
print("     Fix: LITE_MODE = False, more epochs. Do NOT lower the gates.")
print("  ALL edges collapsed into ONE class -> degenerate model, expected under")
print("     LITE_MODE with only 2 capture days. Not a code fault.")
print("  edges pass the gate but still no rules -> corroboration or allowlist")
print("     filtered them; inspect SDNTranslator(require_node_corroboration).")

## 10 — Optional: zero-day evaluation

Retrains with one attack family removed **entirely**, then measures whether the
model flags it. Run this only when you have time — it is a second full training
run into a separate folder so it cannot touch your main checkpoints.

**Read `docs/OPEN_SET.md` before quoting any number from this.** Current honest
position: the model detects an unseen family inconsistently, and the post-hoc
Mahalanobis scorer is *worse* than the plain baseline. This measures the gap; it
does not close it.

In [ ]:
# WHY: measure generalisation to an attack never seen in training. Writes to a
# SEPARATE base_dir so your main run's checkpoints are untouched.
RUN_ZERO_DAY = False
HOLDOUT      = "Botnet"

if RUN_ZERO_DAY:
    from graphsentinel.config import Config
    from graphsentinel.train import train

    zc = Config.from_dict(cfg.to_dict())
    zc.base_dir              = os.path.join(BASE, f"zeroday_{HOLDOUT}")
    zc.data.processed_dir    = f"/content/gs_zd_{HOLDOUT}/"
    zc.data.split_strategy   = "attack_holdout"
    zc.data.holdout_attacks  = [HOLDOUT]
    zc.train.epochs          = 25
    zc.ensure_dirs()

    zrep = train(zc, resume=True, force_rebuild=False, verbose=True)

    os_ = zrep.get("open_set", {})
    print(f"\nOPEN-SET (held out: {HOLDOUT})")
    print(f"  {'score':<30s} {'OSCR':>8s} {'AUROC':>8s} {'FPR@95':>8s}")
    for name, mm in os_.items():
        if isinstance(mm, dict) and "oscr_auc" in mm:
            print(f"  {name:<30s} {mm['oscr_auc']:8.4f} "
                  f"{mm.get('ood_auroc', float('nan')):8.4f} {mm['fpr_at_95tpr']:8.4f}")
    ha = os_.get("_head_assignment_of_unknown")
    if ha:
        print(f"\n  the head sends the unseen {HOLDOUT} to: {ha}")
        print("  -> into BENIGN = 'collapse' (bad). Into another attack = 'shoehorn'.")
else:
    print("skipped (RUN_ZERO_DAY = False)")

## 11 — Where this leaves you

### Safe to build on now
- Host-level detection with a real IP for every prediction.
- Per-flow verdicts that produce exact OpenFlow matches.
- A stable JSON contract (`model_card.json`) so the backend never hard-codes
  feature counts.

### Do NOT claim yet
- **Zero-day detection.** Defensible wording: *"detects known attack families
  and produces an anomaly score for unfamiliar traffic; unknown-family detection
  is inconsistent and under evaluation."*
- **Real-time performance**, until Section 9 has been run on Mininet/Scapy
  traffic rather than replayed CSV.
- **Autonomous blocking.** Keep `dry_run=True` until the false-positive rate is
  measured on *your* traffic and an allowlist is configured.

### Before Mininet integration
CICIDS2017 and a Mininet lab differ in host count, topology, and traffic mix.
The model is inductive — it consumes any host graph, and unseen IPs are fine —
but **feature distributions will shift**, which is what the EMA scaler and its
drift alarm exist to absorb. Expect to retrain or fine-tune on Mininet-generated
traffic; do not assume CICIDS2017 weights transfer cleanly.

### Recovery
| situation | do this |
|---|---|
| runtime crashed mid-training | re-run cells 1→6, then the training cell (`resume=True`) |
| loss went NaN / results look wrong | RESET cell: `RESET_CHECKPOINTS` + `RESET_LOGS` |
| changed window size or split | RESET cell: `RESET_GRAPH_CACHE` + `RESET_PROCESSED` |
| changed dataset files | RESET cell: `RESET_PROCESSED` |
| out of RAM repeatedly | `LITE_MODE = True`, raise `window_seconds`, lower `max_edges_per_graph` |
| edited package on Drive | re-run the copy-code cell (cell 4) |

In [ ]:
# WHY: last cell. Confirms the deliverables exist on Drive so you know the run
# actually produced something usable before you close the tab.
from pathlib import Path
print("GraphSentinel run summary")
print("=" * 66)
print("base:", BASE)
for label, path, keys in [
    ("checkpoints", cfg.checkpoint_path, ["best.pt", "last.pt"]),
    ("models",      cfg.model_path,      ["weights.pt", "model_card.json"]),
    ("logs",        cfg.log_path,        ["training_log.csv", "test_report.json"]),
]:
    print(f"\n{label}:")
    for k in keys:
        p = Path(path) / k
        print(f"  {'OK     ' if p.exists() else 'MISSING'} {k}")

try:
    print(f"\nheadline: node macro F1 = {rep['metrics'].get('node_macro_f1', float('nan')):.4f}"
          f" | recall@1%FPR = {rep['metrics'].get('node_recall_at_fpr_0.01', float('nan')):.4f}")
    print(f"split protocol: {cfg.data.split_strategy}  (this qualifies the number above)")
except Exception:
    pass
print("\nNext: hand models/ + docs/BACKEND_CONTRACT.md to the backend.")